# Appendix · Quantifying the shape of mean firing-rate traces

The chapter summarises each unit's temporal structure with a single number,
**Dominant-Peak Prominence (DPP)**. That is a deliberate simplification: DPP asks
only *"is there one tall peak with no comparable rival?"* and is silent about how
**wide** the response is and how **ragged** the trace is between peaks.

This appendix reports the fuller set of trace statistics, shows their distributions,
and establishes what DPP does and does not capture.

**Scope.** All panels use the **fixation-category-modulated** subpopulation only. A
width or raggedness statistic computed on an unmodulated trace describes the noise
floor rather than a response, so including those units would blur every distribution
towards the same shape.

Every figure is a single embeddable panel (one row of four regions), written as an
Illustrator-editable PDF and a 400 dpi PNG.

## A1 · Setup

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = next(parent for parent in Path.cwd().parents if (parent / "src").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from dal_monte_2022_analysis.config.load import load_config
from dal_monte_2022_analysis.ephys.plotting import thesis_common as style
from dal_monte_2022_analysis.ephys.plotting import thesis_single_unit as figs
from dal_monte_2022_analysis.ephys.plotting import thesis_single_unit_data as data
from dal_monte_2022_analysis.ephys.plotting import thesis_chapter_text as text
from dal_monte_2022_analysis.ephys.plotting import thesis_metric_space as space
from dal_monte_2022_analysis.ephys.plotting import thesis_trace_metrics as metrics
from dal_monte_2022_analysis.ephys.analysis.fixation_peakiness import (
    decompose_dominant_peak_prominence,
)

DATASET_CFG_PATH = repo_root / "configs" / "dataset.yaml"
PLOTTING_CFG_PATH = repo_root / "configs" / "plotting.yaml"
PSTH_CFG_PATH = repo_root / "configs" / "ephys_fixation_psth.yaml"

dataset_cfg = load_config(DATASET_CFG_PATH)
plotting_cfg = load_config(PLOTTING_CFG_PATH)
psth_cfg = load_config(PSTH_CFG_PATH)

style.apply_thesis_plot_style(plotting_cfg)
pd.set_option("display.width", 190)
pd.set_option("display.max_columns", 60)

ANALYSIS_ROOT = data.resolve_psth_analysis_root(dataset_cfg)
FIGURE_SETTINGS = style.ThesisFigureSettings(output_dir=ANALYSIS_ROOT / "single_unit_thesis")
FIGURE_SETTINGS.output_dir.mkdir(parents=True, exist_ok=True)

FIGURE_MANIFEST: dict[str, dict[str, Path]] = {}

In [ ]:
units = data.load_thesis_unit_table(ANALYSIS_ROOT)
selective = data.load_trace_metric_table(ANALYSIS_ROOT, units, selective_only=True)

exemplars = data.build_exemplar_table(units, data.parse_config_exemplar_map(psth_cfg))
exemplar_metrics = data.load_trace_metric_table(
    ANALYSIS_ROOT, exemplars, selective_only=False
)

FIGURE_SETTINGS = style.ThesisFigureSettings(
    output_dir=ANALYSIS_ROOT / "single_unit_thesis" / "appendix"
)
FIGURE_SETTINGS.output_dir.mkdir(parents=True, exist_ok=True)

print(f"{len(selective)} fixation-category-modulated units with trace metrics")
print(f"figures -> {FIGURE_SETTINGS.output_dir}")

## A2 · The metrics

Each metric is computed on the condition-average firing-rate trace over −500 to
+500 ms, with the baseline taken as the 10th percentile of the in-window trace. They
are grouped by the property they measure, so it is clear which are alternative
estimates of the same thing and which are genuinely independent.

In [ ]:
METRIC_ORDER = [
    "mass_width_frac_50",
    "effective_width_ms",
    "lifetime_sparseness",
    "peak_dominance",
    "n_prominent_peaks",
    "fwhm_frac",
    "sustained_frac",
    "roughness",
    "autocorr_width_ms",
    "peak_z",
]
definitions = metrics.build_metric_definition_table(METRIC_ORDER)
display(definitions)

## A3 · Distribution of each metric

One panel per metric, four regions per panel, with the chapter's high- and low-DPP
example units marked. Bin edges are shared across regions so the panel shapes can be
compared directly; the axis is trimmed at the 99.5th percentile because several
metrics have a thin right tail that would otherwise flatten the body.

In [ ]:
FIGURE_MANIFEST = {}
for index, metric in enumerate(METRIC_ORDER, start=1):
    fig = metrics.plot_metric_distribution_panel(
        selective, metric, exemplars=exemplar_metrics
    )
    stem = f"figA{index:02d}_{metric}"
    FIGURE_MANIFEST[stem] = style.save_thesis_figure(fig, FIGURE_SETTINGS, stem)
    display(Markdown(f"**{metrics.metric_axis_label(metric)}**"))
    display(Image(data=style.figure_to_png_bytes(fig)))

In [ ]:
metric_summary = metrics.build_metric_summary_table(selective, metrics=METRIC_ORDER)
display(metric_summary.round(3))

## A4 · Which metrics are redundant

Several of these measure the same underlying property by different routes. The
rank-correlation matrix below identifies those groups, so only one member of each
needs to be reported in the chapter.

In [ ]:
CORRELATION_METRICS = METRIC_ORDER + [style.DPP_COLUMN]
fig, correlation = metrics.plot_metric_correlation_panel(
    selective, metrics=CORRELATION_METRICS
)
FIGURE_MANIFEST["figA11_metric_correlation"] = style.save_thesis_figure(
    fig, FIGURE_SETTINGS, "figA11_metric_correlation"
)
display(Image(data=style.figure_to_png_bytes(fig)))
display(correlation.round(2))

## A5 · What DPP does and does not capture

Each panel plots one trace metric against DPP within region. A metric that tracks DPP
closely is redundant with it; one that is orthogonal describes a property of the same
trace that the chapter's single score discards.

In [ ]:
RELATIONSHIP_METRICS = ["peak_dominance", "fwhm_frac", "roughness", "peak_z"]
relationship_tables = []
for index, metric in enumerate(RELATIONSHIP_METRICS, start=12):
    fig, table = metrics.plot_metric_vs_dpp_panel(
        selective, metric, exemplars=exemplar_metrics
    )
    stem = f"figA{index:02d}_{metric}_vs_dpp"
    FIGURE_MANIFEST[stem] = style.save_thesis_figure(fig, FIGURE_SETTINGS, stem)
    display(Markdown(f"**{metrics.metric_axis_label(metric)} vs {style.DPP_ABBREV}**"))
    display(Image(data=style.figure_to_png_bytes(fig)))
    relationship_tables.append(table)

relationships = pd.concat(relationship_tables, ignore_index=True)
display(relationships.round(4))

## A6 · Where the example units fall on every metric

Percentile rank within each unit's own region. This is the audit of the chapter's
example selection: a high-DPP example should sit high on peak dominance and low on
the width measures, and a low-DPP example should do the opposite.

In [ ]:
rank_table = metrics.build_exemplar_metric_rank_table(
    selective, exemplar_metrics, metrics=RELATIONSHIP_METRICS + ["mass_width_frac_50"]
)
display(rank_table.round(2))

## A7 · Persist tables and figure manifest

In [ ]:
exports = {
    "trace_metric_definitions.csv": definitions,
    "trace_metric_summary_by_region.csv": metric_summary,
    "trace_metric_correlation.csv": correlation.reset_index(names="metric"),
    "trace_metric_vs_dpp.csv": relationships,
    "exemplar_metric_percentiles.csv": rank_table,
}
for filename, frame in exports.items():
    frame.to_csv(FIGURE_SETTINGS.output_dir / filename, index=False)
    print(f"wrote {filename:40s} ({len(frame)} rows)")

print()
for stem in FIGURE_MANIFEST:
    print(stem)

## Appendix summary

- The metrics fall into a small number of correlated groups (A4): width measures
  (`mass_width_frac_50`, `effective_width_ms`, `fwhm_frac`, `sustained_frac`,
  `lifetime_sparseness`) are largely interchangeable, as are the two composites.
  `roughness` and `peak_dominance` carry information the width measures do not.
- **DPP measures peak isolation, not response width** (A5). Its strongest
  association is with `peak_dominance` (rho approximately 0.3) and its next with
  `roughness` and the prominent-peak count (both approximately -0.3); it is close to
  independent of every width measure (|rho| < 0.15 against `mass_width_frac_50`,
  `effective_width_ms`, `fwhm_frac` and `sustained_frac`). The chapter's single score
  should therefore not be described as measuring how *narrow* a response is - it says
  a peak stands alone, not that it is brief. Note also that no single metric explains
  DPP: even the strongest correlation is modest, because DPP combines peak height and
  rival suppression in a way none of the individual statistics reproduces.
- The high-DPP example units rank at the top of their region on peak dominance and at
  the bottom on the width measures, as their label claims (A6). Among the low-DPP
  examples, **dmPFC 1516 does not**: it ranks high on peak dominance and low on FWHM,
  behaving like a high-DPP unit on the shape metrics even though its DPP score is
  only mid-range. Its high `roughness` is what holds its DPP down.